In [4]:
import pandas as pd
import numpy as np
import os
import collections
import pysam
from Bio.Seq import Seq
from Bio import pairwise2
from Bio.pairwise2 import format_alignment
from scipy.spatial import distance
from tqdm import tqdm

In [5]:
goodElements=['Retroposon/SVA','SINE/Alu','LINE/L1']
ElementDict={x:{} for x in goodElements}

In [6]:
import json
with open("/LeeLab/HPRC/chromosomeY/Data/MEIs/allElement_100Match_MEIs.09152025.json", "r") as f:
    loaded_dict = json.load(f)

In [12]:
directory="/LeeLab/HPRC/chromosomeY/Data/MEIs/part2_limeaid/limeaid_MEI_out"
fullList=[]
for file in os.listdir(directory):
    if '.tsv' in file:
        limeaid = pd.read_csv(directory+"/"+file,sep='\t')
        goodElements=['Retroposon/SVA','SINE/Alu','LINE/L1']
        nonElementDF = limeaid[~limeaid['Element_Designation'].isin(goodElements)].copy()
        ElementDF = limeaid[limeaid['Element_Designation'].isin(goodElements)].copy()
        decentINS = ElementDF[(ElementDF['FLAGS']=='No_Flags') & (ElementDF['Tail_Type']!='No_Tail_Type')].copy()
        columnList = decentINS.columns

        for row in decentINS.index:
            fullList.append([decentINS.at[row,column] for column in columnList])
    else:
        continue

In [31]:
fullListDF = pd.DataFrame(data=fullList, columns = columnList)

In [32]:
fullListDF['AlignmentGrouping']='NONE'

In [33]:
fullListDF.set_index("ID", inplace=True)

In [34]:
fullListDF.head(2)

,Sequence,Element_Hits,Sequence_Length,Element_Designation,Element_Annotation,Element_Proportion,Element_Percentage,Element_Divergence,Orientation,FLAGS,Tail_Begins,Tail_Type,Tail_Length,Tail_Seed_Hits,Unique_Element_Count,Twin_Priming_Flag,AlignmentGrouping
ID,,,,,,,,,,,,,,,,,
HG00731_chrY:7998307-7999190,CCCTTAACATTTTTTCCTTCATTTAAACTTTGACACCTCACACGGC...,['6607 5.3 0.1 0.1 HG00731_chrY:7998307-799919...,884,LINE/L1,L1HS,{'L1HS': 0.9389140271493213},0.938914,5.3,+,No_Flags,9,Possible_A-Tail_and_Possible_T-Tail*,6,8,One_Element,NONE,NONE
HG00731_chrY:54488600-54488966,CCCTGGTATTTTTTTAAGAATGCTATTGGCGGCCGGGCGCGGTGGC...,['2821 1.6 0.3 0.0 HG00731_chrY:54488600-54488...,367,SINE/Alu,AluY,{'AluY': 0.8365122615803815},0.836512,1.6,+,No_Flags,33,Possible_A-Tail*_and_Possible_T-Tail,24,22,One_Element,NONE,NONE


In [36]:
from Bio import SeqIO
import re

fasta_sequences = SeqIO.parse(open("/LeeLab/HPRC/chromosomeY/Data/MEIs/part3_elements/FullMatch_Groups/LINE_L1_youngElements_MarkEdited.09182025.afa"),'fasta')
LineGroup=1
GroupMember=1
for fasta in fasta_sequences:
    name, sequence = fasta.id, str(fasta.seq)
    if name == 'New_sequence':
        LineGroup+=1
        GroupMember=1
    else:
        clean_name = re.split(r"-L1", name)[0]
        fullListDF.at[clean_name,'AlignmentGrouping'] = "L1-"+str(LineGroup)+"-"+str(GroupMember)
        for member in loaded_dict['LINE/L1'][name]['Group']:
            clean_name2 = re.split(r"-L1", member)[0]
            fullListDF.at[clean_name2,'AlignmentGrouping'] = "L1-"+str(LineGroup)+"-"+str(GroupMember)
        GroupMember+=1

In [46]:
fasta_sequences = SeqIO.parse(open("/LeeLab/HPRC/chromosomeY/Data/MEIs/part3_elements/FullMatch_Groups/Retroposon_SVA_youngElements_aligned_MarkEdited.09152025.fasta"),'fasta')
SVAGroup=1
GroupMember=1
for fasta in fasta_sequences:
    name, sequence = fasta.id, str(fasta.seq)
    if name == 'New_sequence':
        SVAGroup+=1
        GroupMember=1
    else:
        clean_name = re.split(r"-SVA", name)[0]
        fullListDF.at[clean_name,'AlignmentGrouping'] = "SVA-"+str(SVAGroup)+"-"+str(GroupMember)
        for member in loaded_dict['Retroposon/SVA'][name]['Group']:
            clean_name2 = re.split(r"-SVA", member)[0]
            fullListDF.at[clean_name2,'AlignmentGrouping'] = "SVA-"+str(SVAGroup)+"-"+str(GroupMember)
        GroupMember+=1

In [50]:
fasta_sequences = SeqIO.parse(open("/LeeLab/HPRC/chromosomeY/Data/MEIs/part3_elements/FullMatch_Groups/SINE_Alu_youngElements_aligned_MarkGroups.09152025.fasta"),'fasta')
AluGroup=1
GroupMember=1
for fasta in fasta_sequences:
    name, sequence = fasta.id, str(fasta.seq)
    if name == 'New_sequence':
        AluGroup+=1
        GroupMember=1
    else:
        clean_name = re.split(r"-Alu", name)[0]
        fullListDF.at[clean_name,'AlignmentGrouping'] = "Alu-"+str(AluGroup)+"-"+str(GroupMember)
        for member in loaded_dict['SINE/Alu'][name]['Group']:
            clean_name2 = re.split(r"-Alu", member)[0]
            fullListDF.at[clean_name2,'AlignmentGrouping'] = "Alu-"+str(AluGroup)+"-"+str(GroupMember)
        GroupMember+=1

In [53]:
filledElementDF = fullListDF[fullListDF['AlignmentGrouping']!='NONE'].copy()

In [55]:
filledElementDF['Alignment_MainGroup']=[x.split("-")[1] for x in filledElementDF['AlignmentGrouping']]
filledElementDF['Alignment_SubGroup']=[x.split("-")[2] for x in filledElementDF['AlignmentGrouping']]

In [57]:
#filledElementDF.to_csv("/LeeLab/HPRC/chromosomeY/Data/MEIs/non-LTR_retrotransposons_GroupFilled.09182025.csv")